<a href="https://colab.research.google.com/github/kodihasini/2520090209_DeepLearning/blob/main/DeepLearning_Skill2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# Install Triton HTTP/gRPC client
!pip install tritonclient[all] fastapi uvicorn pydantic pillow torchvision

import os
# Create model repository layout for a TensorRT or ONNX model
os.makedirs("model_repository/vision_model/1", exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.8/616.8 kB 4.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 7.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 9.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 10.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 12.0 MB/s  0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml_dtypes 0.6.0
    Uninstalling ml_dtypes-0.6.0:
      Successfully uninstalled ml_dtypes-0.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [cuda-python]


In [3]:
config_content = """
name: "vision_model"
platform: "onnxruntime_onnx"
max_batch_size: 8
input [
  {
    name: "input"
    data_type: TYPE_FP32
    dims: [ 3, 224, 224 ]
  }
]
output [
  {
    name: "output"
    data_type: TYPE_FP32
    dims: [ 1000 ]
  }
]
"""

with open("model_repository/vision_model/config.pbtxt", "w") as f:
    f.write(config_content)

In [4]:
import torch
import torchvision.models as models

# Load a pretrained vision model
model = models.resnet50(pretrained=True).eval()

# Dummy input matching the config dimensions
dummy_input = torch.randn(1, 3, 224, 224)

# Export to ONNX
torch.onnx.export(
    model,
    dummy_input,
    "model_repository/vision_model/1/model.onnx",
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 126MB/s]


ModuleNotFoundError: No module named 'onnxscript'

In [ ]:
# Download Triton Server release binaries for Linux (using -L to follow redirects)
!curl -L -o tritonserver24.03-py3.tgz https://github.com/triton-inference-server/server/releases/download/v24.03-rel/tritonserver24.03-py3.tgz

import os
import subprocess
import time

# Ensure the archive exists before extracting
if os.path.exists("tritonserver24.03-py3.tgz"):
    print("Tarball downloaded successfully. Extracting...")
    subprocess.run(["tar", "-xzf", "tritonserver24.03-py3.tgz"])
else:
    raise FileNotFoundError("Failed to download Triton Server tarball!")

# Start Triton server in the background pointing to your model repository
if os.path.exists("./bin/tritonserver"):
    print("Starting Triton Server...")
    triton_proc = subprocess.Popen([
        "./bin/tritonserver",
        "--model-repository=./model_repository",
        "--http-port=8000",
        "--grpc-port=8001"
    ])
    # Wait a few seconds for Triton to spin up
    time.sleep(5)
    print("Triton Server process started.")
else:
    raise FileNotFoundError("Triton server binary not found at ./bin/tritonserver after extraction.")

In [ ]:
%%writefile app.py
from fastapi import FastAPI, UploadFile, File
import tritonclient.http as httpclient
import numpy as np
from PIL import Image
import io

app = FastAPI(title="VisionPort API")

# Connect to local Triton instance
triton_client = httpclient.InferenceServerClient(url="localhost:8000")

@app.get("/health")
def health_check():
    return {"status": "running", "triton_live": triton_client.is_server_live()}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    # 1. Read and preprocess image
    image_data = await file.read()
    image = Image.open(io.BytesIO(image_data)).convert("RGB")
    image = image.resize((224, 224))

    # Convert to CHW format and normalize (Standard ImageNet preprocessing)
    img_array = np.array(image, dtype=np.float32) / 255.0
    img_array = np.transpose(img_array, (2, 0, 1)) # [C, H, W]
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension [1, 3, 224, 224]

    # 2. Setup Triton input/output tensors
    inputs = [httpclient.InferInput("input", img_array.shape, "FP32")]
    inputs[0].set_data_from_numpy(img_array)

    outputs = [httpclient.InferRequestedOutput("output")]

    # 3. Send inference request to Triton
    response = triton_client.infer(model_name="vision_model", inputs=inputs, outputs=outputs)
    result = response.as_numpy("output")

    return {
        "prediction_shape": result.shape,
        "top_class_index": int(np.argmax(result))
    }

Writing app.py


In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_fastapi():
    uvicorn.run("app:app", host="0.0.0.0", port=8080)

threading.Thread(target=run_fastapi, daemon=True).start()